# Train OSNet on MEVID with Kaggle

Enable **GPU** and **Internet**, then run every cell from top to bottom.
MEVID is checked, downloaded when absent, extracted, and validated before FastReID is installed or configured.
`/kaggle/input` is always treated as read-only.

## 1. Environment setup and experiment settings

In [ ]:
from pathlib import Path
import os
import sys
import json
import shutil
import subprocess
import importlib.util

import torch
import torchvision

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in Kaggle Settings > Accelerator, restart the session, then Run All.")

WORK = Path("/kaggle/working/osnet_mevid")
OUT = WORK / "results/osnet"
SEARCH_LOCATIONS = [
    Path("/kaggle/input"),
    Path("/kaggle/working/data"),
    Path("/kaggle/temp"),
]
PREFERRED_MEVID_ROOT = Path("/kaggle/working/data/mevid")
TEMP_MEVID_ROOT = Path("/kaggle/temp/mevid")

EPOCHS = 60
BATCH_SIZE = 64
EVAL_EVERY = 10
NUM_WORKERS = 2
FRAME_STEP = 20
SEED = 42
LOG_EVERY = 10
DELETE_ARCHIVES_AFTER_EXTRACTION = True

PRETRAINED_WEIGHTS = ""
RESUME_CHECKPOINT = ""
AUTO_RESUME = True

assert EPOCHS >= 1
assert BATCH_SIZE >= 8 and BATCH_SIZE % 4 == 0
assert FRAME_STEP >= 1 and EVAL_EVERY >= 1 and NUM_WORKERS >= 0
WORK.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
print("PyTorch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0))

## 2. Check whether an extracted MEVID dataset already exists

Search order: `/kaggle/input`, `/kaggle/working/data`, then `/kaggle/temp`.
The first parent containing all three required folders is selected. Nothing is written to `/kaggle/input`.

In [ ]:
REQUIRED_MEVID_FOLDERS = (
    "bbox_train",
    "bbox_test",
    "mevid-v1-annotation-data",
)


def locate_mevid_root(search_locations):
    required = {name.casefold(): name for name in REQUIRED_MEVID_FOLDERS}
    for search_root in search_locations:
        search_root = Path(search_root)
        if not search_root.is_dir():
            continue
        for parent, dirs, _ in os.walk(search_root):
            by_name = {name.casefold(): name for name in dirs}
            if all(name in by_name for name in required):
                root = Path(parent).resolve()
                return root
            # Never descend into the large MEVID image trees while searching.
            if Path(parent).name.casefold() in required:
                dirs[:] = []
    return None


MEVID_ROOT = locate_mevid_root(SEARCH_LOCATIONS)
if MEVID_ROOT is None:
    print("Extracted MEVID was not found; download is required.")
else:
    print("Using existing MEVID:", MEVID_ROOT)

## 3. Download MEVID when missing

Official archives are downloaded only when an extracted dataset was not found. Existing complete archives are
reused, and `.part` downloads resume. The destination is `/kaggle/working/data/mevid` when space permits;
otherwise `/kaggle/temp/mevid` is used. Download failure stops the notebook immediately.

In [ ]:
import time
import urllib.request

MEVID_BASE_URL = "https://mevadata-public-01.s3.amazonaws.com/mevid-annotations"
MEVID_ARCHIVES = {
    "mevid-v1-annotation-data": "mevid-v1-annotation-data.zip",
    "bbox_train": "mevid-v1-bbox-train.tgz",
    "bbox_test": "mevid-v1-bbox-test.tgz",
}


def remote_size(url):
    request = urllib.request.Request(url, method="HEAD")
    with urllib.request.urlopen(request, timeout=60) as response:
        return int(response.headers.get("Content-Length", 0))


def select_writable_root(total_archive_bytes):
    # JPEG tarballs expand only moderately, but extraction and archives overlap temporarily.
    estimated_required = max(int(total_archive_bytes * 1.55), 50 * 1024**3)
    candidates = (PREFERRED_MEVID_ROOT, TEMP_MEVID_ROOT)
    availability = []
    for candidate in candidates:
        candidate.parent.mkdir(parents=True, exist_ok=True)
        free = shutil.disk_usage(candidate.parent).free
        availability.append((candidate, free))
        if free >= estimated_required:
            print(
                f"Selected {candidate} ({free / 1024**3:.1f} GiB free; "
                f"estimated requirement {estimated_required / 1024**3:.1f} GiB)"
            )
            return candidate
    detail = ", ".join(f"{path}: {free / 1024**3:.1f} GiB free" for path, free in availability)
    raise RuntimeError(
        "MEVID is missing and writable storage is insufficient for automatic preparation. "
        f"Available: {detail}. Attach an extracted MEVID Kaggle Dataset under /kaggle/input."
    )


def download_with_resume(url, destination):
    destination = Path(destination)
    partial = destination.with_suffix(destination.suffix + ".part")
    expected = remote_size(url)
    if destination.is_file() and (not expected or destination.stat().st_size == expected):
        print(f"Reusing downloaded archive: {destination}")
        return destination
    if destination.exists():
        destination.unlink()

    offset = partial.stat().st_size if partial.exists() else 0
    headers = {"Range": f"bytes={offset}-"} if offset else {}
    request = urllib.request.Request(url, headers=headers)
    started = time.monotonic()
    with urllib.request.urlopen(request, timeout=120) as response:
        append = offset > 0 and getattr(response, "status", None) == 206
        mode = "ab" if append else "wb"
        downloaded = offset if append else 0
        if not append:
            offset = 0
        next_report = downloaded + 512 * 1024**2
        with partial.open(mode) as output:
            while True:
                chunk = response.read(1024 * 1024)
                if not chunk:
                    break
                output.write(chunk)
                downloaded += len(chunk)
                if downloaded >= next_report:
                    speed = (downloaded - offset) / max(time.monotonic() - started, 0.001)
                    print(
                        f"{destination.name}: {downloaded / 1024**3:.1f}/"
                        f"{expected / 1024**3:.1f} GiB, {speed / 1024**2:.1f} MiB/s",
                        flush=True,
                    )
                    next_report += 512 * 1024**2
    if expected and partial.stat().st_size != expected:
        raise RuntimeError(
            f"Download incomplete for {destination.name}: "
            f"{partial.stat().st_size} of {expected} bytes. Rerun this cell to resume."
        )
    partial.replace(destination)
    return destination


ARCHIVE_PATHS = {}
if MEVID_ROOT is None:
    try:
        archive_sizes = {
            folder: remote_size(f"{MEVID_BASE_URL}/{filename}")
            for folder, filename in MEVID_ARCHIVES.items()
        }
        MEVID_DOWNLOAD_ROOT = select_writable_root(sum(archive_sizes.values()))
        MEVID_DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
        ARCHIVE_DIR = MEVID_DOWNLOAD_ROOT.parent / "mevid_archives"
        ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

        for folder, filename in MEVID_ARCHIVES.items():
            marker = MEVID_DOWNLOAD_ROOT / f".{filename}.extracted"
            if marker.is_file() and (MEVID_DOWNLOAD_ROOT / folder).is_dir():
                print(f"{folder}: extraction already completed")
                continue
            ARCHIVE_PATHS[folder] = download_with_resume(
                f"{MEVID_BASE_URL}/{filename}",
                ARCHIVE_DIR / filename,
            )
    except Exception as error:
        raise RuntimeError(f"MEVID download failed; training has been stopped: {error}") from error
else:
    print("Download skipped because extracted MEVID already exists.")

## 4. Extract downloaded MEVID archives

Extraction is idempotent. Completed folders have marker files and are skipped. Downloaded archives are retained
if extraction fails, so rerunning reuses them; successful archives can be removed to recover writable storage.

In [ ]:
import tarfile
import zipfile


def checked_output_path(root, member_name):
    root = Path(root).resolve()
    destination = (root / member_name).resolve()
    if destination != root and root not in destination.parents:
        raise RuntimeError(f"Unsafe archive path: {member_name}")
    return destination


def extract_archive(archive_path, destination):
    archive_path = Path(archive_path)
    destination = Path(destination)
    if archive_path.suffix == ".zip":
        with zipfile.ZipFile(archive_path) as archive:
            for member in archive.infolist():
                checked_output_path(destination, member.filename)
                archive.extract(member, destination)
    else:
        with tarfile.open(archive_path, "r:gz") as archive:
            for member in archive:
                checked_output_path(destination, member.name)
                if member.issym() or member.islnk():
                    raise RuntimeError(f"Archive links are not allowed: {member.name}")
                archive.extract(member, destination)


if MEVID_ROOT is None:
    try:
        for folder, filename in MEVID_ARCHIVES.items():
            marker = MEVID_DOWNLOAD_ROOT / f".{filename}.extracted"
            folder_path = MEVID_DOWNLOAD_ROOT / folder
            if marker.is_file() and folder_path.is_dir():
                print(f"{folder}: already extracted")
                continue
            archive_path = ARCHIVE_PATHS.get(folder, ARCHIVE_DIR / filename)
            if not archive_path.is_file():
                raise FileNotFoundError(f"Missing downloaded archive: {archive_path}")
            print(f"Extracting {archive_path.name} ...", flush=True)
            extract_archive(archive_path, MEVID_DOWNLOAD_ROOT)
            if not folder_path.is_dir():
                raise RuntimeError(f"Archive did not create required folder: {folder_path}")
            marker.touch()
            if DELETE_ARCHIVES_AFTER_EXTRACTION:
                archive_path.unlink()
            print(f"{folder}: extraction complete")
        MEVID_ROOT = locate_mevid_root([MEVID_DOWNLOAD_ROOT])
    except Exception as error:
        raise RuntimeError(f"MEVID extraction failed; training has been stopped: {error}") from error

if MEVID_ROOT is None:
    raise RuntimeError("MEVID preparation failed; training has been stopped.")

## 5. Validate all required MEVID folders

In [ ]:
required_paths = {
    "bbox_train": MEVID_ROOT / "bbox_train",
    "bbox_test": MEVID_ROOT / "bbox_test",
    "annotation": MEVID_ROOT / "mevid-v1-annotation-data",
}
for name, path in required_paths.items():
    if not path.is_dir():
        raise RuntimeError(f"Missing dataset directory: {name}: {path}")

required_annotations = (
    "train_name.txt",
    "test_name.txt",
    "track_train_info.txt",
    "track_test_info.txt",
    "query_IDX.txt",
)
for filename in required_annotations:
    path = required_paths["annotation"] / filename
    if not path.is_file():
        raise RuntimeError(f"Missing MEVID annotation file: {path}")

print("MEVID folder validation passed.")

## 6. Set `DATA_DIRS` and count images

In [ ]:
DATA_DIRS = {name: str(path.resolve()) for name, path in required_paths.items()}

for name, path in DATA_DIRS.items():
    assert Path(path).exists(), f"Missing dataset directory: {name}: {path}"


def count_images(directory):
    extensions = {".jpg", ".jpeg", ".png", ".bmp"}
    count = 0
    for _, _, files in os.walk(directory):
        count += sum(Path(filename).suffix.casefold() in extensions for filename in files)
    return count


TRAIN_IMAGE_COUNT = count_images(DATA_DIRS["bbox_train"])
TEST_IMAGE_COUNT = count_images(DATA_DIRS["bbox_test"])
if TRAIN_IMAGE_COUNT == 0 or TEST_IMAGE_COUNT == 0:
    raise RuntimeError(
        f"MEVID image folders are empty: train={TRAIN_IMAGE_COUNT}, test={TEST_IMAGE_COUNT}"
    )

DATA_ROOT = MEVID_ROOT
print("MEVID dataset ready")
print("bbox_train:", DATA_DIRS["bbox_train"])
print("bbox_test:", DATA_DIRS["bbox_test"])
print("annotation:", DATA_DIRS["annotation"])
print(f"bbox_train images: {TRAIN_IMAGE_COUNT:,}")
print(f"bbox_test images: {TEST_IMAGE_COUNT:,}")

## 7. Install FastReID and OSNet dependencies

This cell is intentionally after MEVID validation. No FastReID configuration or training runs unless all dataset
preparation cells above finish successfully.

In [ ]:
packages = {
    "yacs": "yacs",
    "termcolor": "termcolor",
    "prettytable": "prettytable",
    "easydict": "easydict",
    "gdown": "gdown",
    "faiss": "faiss-cpu",
    "tensorboard": "tensorboard",
    "tabulate": "tabulate",
    "tqdm": "tqdm",
    "sklearn": "scikit-learn",
    "yaml": "PyYAML",
    "scipy": "scipy",
}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

FASTREID_COMMIT = "c9bc3ceb2f7a6438b62fb515ea3df6d1e999e95d"
FASTREID_ROOT = WORK / "fast-reid/upstream"
if not (FASTREID_ROOT / ".git").exists():
    FASTREID_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--quiet", "https://github.com/JDAI-CV/fast-reid.git", str(FASTREID_ROOT)],
        check=True,
    )
subprocess.run(
    ["git", "-C", str(FASTREID_ROOT), "checkout", "--quiet", "--detach", FASTREID_COMMIT],
    check=True,
)
sys.path.insert(0, str(FASTREID_ROOT))
os.environ["FASTREID_DATASETS"] = str(MEVID_ROOT.parent)
os.environ["TORCH_HOME"] = str(WORK / "weights/torch")

if PRETRAINED_WEIGHTS:
    source = Path(PRETRAINED_WEIGHTS)
    if not source.is_file():
        raise FileNotFoundError(source)
    destination = WORK / "weights/torch/checkpoints/osnet_x1_0_imagenet.pth"
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
print("FastReID ready:", FASTREID_ROOT)

## 8. Build MEVID train/query/gallery splits

In [ ]:
from tqdm.auto import tqdm


def load_mevid_splits(root, frame_step=20):
    annotations = root / "mevid-v1-annotation-data"
    query_rows = {int(float(value)) for value in (annotations / "query_IDX.txt").read_text().split()}
    splits = {"train": [], "query": [], "gallery": []}

    for subset in ("train", "test"):
        names = (annotations / f"{subset}_name.txt").read_text().splitlines()
        tracks = (annotations / f"track_{subset}_info.txt").read_text().splitlines()
        for row, line in enumerate(tqdm(tracks, desc=f"Reading {subset}", unit="track")):
            start, end, pid, outfit, camera = [int(float(value)) for value in line.split()]
            if end == start - 1:
                continue
            if not 0 <= start <= end < len(names):
                raise ValueError(f"Invalid {subset} tracklet {row}: {start}, {end}")
            split = "train" if subset == "train" else ("query" if row in query_rows else "gallery")
            for index in range(start, end + 1, frame_step):
                image = root / f"bbox_{subset}" / f"{pid:04d}" / names[index].strip()
                if not image.is_file():
                    raise FileNotFoundError(image)
                splits[split].append((str(image), pid, camera))

    for name, samples in splits.items():
        if not samples:
            raise RuntimeError(f"MEVID {name} split is empty")
        identities = len({sample[1] for sample in samples})
        print(f"{name:7s}: {len(samples):,} images, {identities} identities")
    return splits


SPLITS = load_mevid_splits(DATA_ROOT, FRAME_STEP)

## 9. Define the FastReID dataset, progress display, and evaluator

In [ ]:
import collections
import collections.abc
collections.Mapping = collections.abc.Mapping
collections.Iterable = collections.abc.Iterable

from fastreid.data.datasets import DATASET_REGISTRY
from fastreid.data.datasets.bases import ImageDataset
from fastreid.engine import DefaultTrainer, hooks
from fastreid.engine.train_loop import HookBase
from fastreid.evaluation import ReidEvaluator


class MEVID_OSNet(ImageDataset):
    def __init__(self, root=None, **kwargs):
        super().__init__(SPLITS["train"], SPLITS["query"], SPLITS["gallery"], **kwargs)


if "MEVID_OSNet" not in DATASET_REGISTRY._obj_map:
    DATASET_REGISTRY.register(MEVID_OSNet)


class NotebookProgress(HookBase):
    bar = None

    def before_epoch(self):
        self.bar = tqdm(
            total=self.trainer.iters_per_epoch,
            desc=f"Epoch {self.trainer.epoch + 1}/{self.trainer.max_epoch}",
            unit="batch",
        )

    def after_step(self):
        latest = self.trainer.storage.latest()
        loss = latest.get("total_loss")
        if loss is not None:
            self.bar.set_postfix(loss=f"{loss[0]:.4f}", refresh=False)
        self.bar.update(1)

    def after_epoch(self):
        if self.bar is not None:
            self.bar.close()

    def after_train(self):
        if self.bar is not None:
            self.bar.close()


class OSNetEvaluator(ReidEvaluator):
    def _compile_dependencies(self):
        pass  # use FastReID's NumPy evaluator; no runtime Cython build


class OSNetTrainer(DefaultTrainer):
    def build_writers(self):
        from fastreid.utils.events import JSONWriter, TensorboardXWriter
        return [
            JSONWriter(os.path.join(self.cfg.OUTPUT_DIR, "metrics.json")),
            TensorboardXWriter(self.cfg.OUTPUT_DIR),
        ]

    def build_hooks(self):
        trainer_hooks = super().build_hooks()
        for hook in trainer_hooks:
            if isinstance(hook, hooks.PeriodicWriter):
                hook._period = LOG_EVERY
        return [NotebookProgress(), *trainer_hooks]

    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_dir=None):
        loader, num_query = cls.build_test_loader(cfg, dataset_name)
        return loader, OSNetEvaluator(cfg, num_query, output_dir)

## 10. Configure OSNet

In [ ]:
from types import SimpleNamespace
from fastreid.config import get_cfg
from fastreid.engine import default_setup
import fastreid.data.build as data_build


# Use a standard DataLoader so worker exceptions are shown directly in Kaggle.
def standard_loader(local_rank, **kwargs):
    kwargs["pin_memory"] = True
    kwargs["num_workers"] = NUM_WORKERS
    return torch.utils.data.DataLoader(**kwargs)


data_build.DataLoaderX = standard_loader

cfg = get_cfg()
cfg.DATASETS.NAMES = ("MEVID_OSNet",)
cfg.DATASETS.TESTS = ("MEVID_OSNet",)
cfg.MODEL.DEVICE = "cuda"
cfg.MODEL.BACKBONE.NAME = "build_osnet_backbone"
cfg.MODEL.BACKBONE.DEPTH = "x1_0"
cfg.MODEL.BACKBONE.FEAT_DIM = 512
cfg.MODEL.BACKBONE.PRETRAIN = True
cfg.MODEL.BACKBONE.PRETRAIN_PATH = ""
cfg.MODEL.BACKBONE.WITH_IBN = False
cfg.MODEL.HEADS.NAME = "EmbeddingHead"
cfg.MODEL.HEADS.NORM = "BN"
cfg.MODEL.HEADS.POOL_LAYER = "GeneralizedMeanPoolingP"
cfg.MODEL.HEADS.EMBEDDING_DIM = 512
cfg.MODEL.LOSSES.NAME = ("CrossEntropyLoss", "TripletLoss")
cfg.MODEL.LOSSES.CE.EPSILON = 0.1
cfg.MODEL.LOSSES.TRI.MARGIN = 0.3
cfg.MODEL.LOSSES.TRI.HARD_MINING = True

cfg.INPUT.SIZE_TRAIN = [256, 128]
cfg.INPUT.SIZE_TEST = [256, 128]
cfg.INPUT.REA.ENABLED = True
cfg.INPUT.REA.PROB = 0.5
cfg.INPUT.FLIP.ENABLED = True
cfg.INPUT.FLIP.PROB = 0.5
cfg.INPUT.PADDING.ENABLED = True
cfg.INPUT.PADDING.SIZE = 10
cfg.INPUT.CJ.ENABLED = True
cfg.INPUT.CJ.PROB = 0.5

cfg.DATALOADER.NUM_INSTANCE = 4
cfg.DATALOADER.NUM_WORKERS = NUM_WORKERS
cfg.DATALOADER.SAMPLER_TRAIN = "BalancedIdentitySampler"
cfg.SOLVER.OPT = "Adam"
cfg.SOLVER.BASE_LR = 0.00035
cfg.SOLVER.WEIGHT_DECAY = 0.0005
cfg.SOLVER.WEIGHT_DECAY_BIAS = 0.0005
cfg.SOLVER.IMS_PER_BATCH = BATCH_SIZE
cfg.SOLVER.MAX_EPOCH = EPOCHS
cfg.SOLVER.WARMUP_ITERS = 2000
cfg.SOLVER.WARMUP_METHOD = "linear"
cfg.SOLVER.STEPS = [40, 55]
cfg.SOLVER.CHECKPOINT_PERIOD = 1

cfg.TEST.EVAL_PERIOD = EVAL_EVERY
cfg.TEST.IMS_PER_BATCH = 128
cfg.TEST.METRIC = "cosine"
cfg.TEST.RERANK.ENABLED = False
cfg.OUTPUT_DIR = str(OUT)
cfg.SEED = SEED
cfg.freeze()

setup_args = SimpleNamespace(config_file="", eval_only=False, resume=AUTO_RESUME)
default_setup(cfg, setup_args)
print(f"OSNet-x1.0 | {EPOCHS} epochs | batch {BATCH_SIZE} | {torch.cuda.get_device_name(0)}")

## 11. Train and evaluate

In [ ]:
last_checkpoint = OUT / "last_checkpoint"
if RESUME_CHECKPOINT and not last_checkpoint.exists():
    source = Path(RESUME_CHECKPOINT)
    if not source.is_file():
        raise FileNotFoundError(source)
    destination = OUT / source.name
    shutil.copy2(source, destination)
    last_checkpoint.write_text(destination.name, encoding="utf-8")

resume = AUTO_RESUME and last_checkpoint.exists()
if last_checkpoint.exists() and not resume:
    raise RuntimeError("A checkpoint exists. Enable AUTO_RESUME or select a new WORK directory.")

trainer = OSNetTrainer(cfg)
trainer.resume_or_load(resume=resume)
print("Resuming latest completed epoch." if resume else "Starting ImageNet-pretrained OSNet training.")
FINAL_METRICS = trainer.train()

result_path = OUT / "evaluation.json"
result_path.write_text(
    json.dumps(FINAL_METRICS, indent=2, default=lambda value: value.item()),
    encoding="utf-8",
)
print("Final evaluation:", result_path)

## 12. Show and download results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import zipfile
from IPython.display import FileLink, display

result_path = OUT / "evaluation.json"
if not result_path.is_file():
    raise RuntimeError("Finish the training cell before opening results.")

metrics = json.loads(result_path.read_text(encoding="utf-8"))
display(pd.DataFrame([metrics], index=["OSNet-x1.0 / MEVID"]))
pd.DataFrame([metrics]).to_csv(OUT / "evaluation.csv", index=False)

history_path = OUT / "metrics.json"
if history_path.is_file():
    records = [json.loads(line) for line in history_path.read_text().splitlines() if line.strip()]
    history = pd.DataFrame(records)
    if {"iteration", "total_loss"}.issubset(history.columns):
        history = history.drop_duplicates("iteration", keep="last").sort_values("iteration")
        ax = history.dropna(subset=["total_loss"]).plot(
            x="iteration", y="total_loss", figsize=(10, 4), grid=True
        )
        ax.set_title("OSNet training loss")
        ax.figure.tight_layout()
        ax.figure.savefig(OUT / "training_loss.png", dpi=150)
        plt.show()

archive = Path("/kaggle/working/osnet_mevid_results.zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in OUT.rglob("*"):
        if path.is_file() and (path.suffix != ".pth" or path.name in {"model_final.pth", "model_best.pth"}):
            bundle.write(path, arcname=str(Path("osnet") / path.relative_to(OUT)))

os.chdir("/kaggle/working")
display(FileLink(archive.name))
display(FileLink(str(result_path.relative_to(Path.cwd()))))
print("All outputs:", OUT)